# KiWIS Feeder (Fabric Notebook)

Runs the generalized KiWIS hydrological feeder against configured KISTERS Web Interoperability Solution endpoints and emits station, timeseries metadata, and observation CloudEvents to the bound Fabric Event Stream.

**Runtime model**
- The `kiwis_kafka` package and generated producer wheels are provided by the per-source Fabric Environment.
- The Event Stream connection string is looked up at runtime via the Fabric Topology API; no secret is stored as a notebook parameter.
- The bridge runs `kiwis-kafka feed --once`, writes state and logs under Lakehouse Files, and exits after one cycle for scheduled execution.


In [ ]:
# === PARAMETERS (overwritten by deploy-feeder-notebook.ps1) ===
EVENTSTREAM_NAME = "kiwis-ingest"
STATE_FILE       = "/lakehouse/default/Files/feeder-state/kiwis/state.json"
POLLING_INTERVAL = 300
RUN_DURATION     = 3300
ONCE_MODE        = True
WORKSPACE_ID     = ""


In [ ]:
import json, os, requests
from notebookutils import mssparkutils as notebookutils

def _get_pbi_token():
    return notebookutils.credentials.getToken('pbi')

def _resolve_workspace_id():
    if WORKSPACE_ID:
        return WORKSPACE_ID
    return notebookutils.runtime.context.get('currentWorkspaceId')

def lookup_es_connection_string(eventstream_name):
    ws = _resolve_workspace_id()
    token = _get_pbi_token()
    headers = {'Authorization': f'Bearer {token}'}
    items = requests.get(f'https://api.fabric.microsoft.com/v1/workspaces/{ws}/items', headers=headers, timeout=60).json().get('value', [])
    es = next(i for i in items if i.get('displayName') == eventstream_name and i.get('type') == 'Eventstream')
    topo = requests.get(f"https://api.fabric.microsoft.com/v1/workspaces/{ws}/eventstreams/{es['id']}/topology", headers=headers, timeout=60).json()
    src = next(s for s in topo.get('sources', []) if s.get('type') in ('CustomEndpoint', 'CustomApp'))
    conn = requests.get(f"https://api.fabric.microsoft.com/v1/workspaces/{ws}/eventstreams/{es['id']}/sources/{src['id']}/connection", headers=headers, timeout=60).json()
    return conn.get('connectionString') or conn.get('primaryConnectionString')

os.environ['CONNECTION_STRING'] = lookup_es_connection_string(EVENTSTREAM_NAME)
os.environ['STATE_FILE'] = STATE_FILE
os.environ['POLLING_INTERVAL'] = str(POLLING_INTERVAL)
os.environ['ONCE_MODE'] = 'true' if ONCE_MODE else 'false'


In [ ]:
import os, sys, threading, traceback
from notebookutils import mssparkutils as notebookutils

LOG_PATH = '/lakehouse/default/Files/feeder-state/kiwis/last-run.log'
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

def _log(message):
    with open(LOG_PATH, 'a', encoding='utf-8') as fh:
        fh.write(str(message) + '\n')

_err = []
def _worker():
    try:
        from kiwis_kafka import app as feeder
        sys.argv = ['kiwis-kafka', 'feed', '--once'] if ONCE_MODE else ['kiwis-kafka', 'feed']
        feeder.main()
    except BaseException as exc:
        _err.append(exc)
        _log(traceback.format_exc())

try:
    _log('Starting KiWIS feeder notebook run')
    t = threading.Thread(target=_worker, daemon=True)
    t.start()
    if ONCE_MODE:
        t.join()
    else:
        t.join(timeout=RUN_DURATION)
        if t.is_alive():
            _log(f'Run duration {RUN_DURATION}s reached; exiting cleanly.')
    if _err:
        raise _err[0]
    notebookutils.notebook.exit('OK')
except BaseException as exc:
    _log(f'FAIL: {exc}')
    notebookutils.notebook.exit('FAIL: ' + str(exc))
